# P06 · Masked-building control (exploratory, development events)

The ring control pools only tokens outside a building, but every token of a Vision Transformer attends to the whole
tile, so ring tokens can still carry information about the building (raised in the readiness review of
22 Nov 2026). This control removes the buildings from the image **before** encoding
(`src/damagesar_bench/controls/masked.py`):

* every footprint in the tile is masked, as drawn and, where the tile has a confident footprint-to-SAR offset (from
  g01), also at the shifted position; both grown by 1 m;
* masked pixels are set to the median of the tile's valid background pixels; no-data stays no-data;
* the masked tile is encoded with DINOv3 ViT-B/16 (native, fp16, as in p02/p03) and pooled over the 10 m ring
  (`ring10m_masked`).

If the masked ring still scores like the building, the context finding does not depend on building information
leaking through attention. Exploratory, added after the freeze; development and secondary events only (confirmatory
labels stay locked here whatever the date, as in r01).

* **Before running:** Runtime → Change runtime type → **T4 GPU**. Secrets: `GH_TOKEN`, `HF_TOKEN`.
  Needs p01 (building table), p02 (DINOv3 ViT-B outline), p03 (unmasked ring, image-free) and g01 (offsets).
* **Time:** about 25–35 minutes (masking + encoding ~20 min, bench run ~5 min); about 1–2 compute units.
* **Outputs:** features in `02_models/features/<date>_masked_ring/`, the run in this run's folder.

In [ ]:
NOTEBOOK_NAME = "p06_masked_building_control"

# Drive layout (FYP Google account). Keep in sync with src/damagesar_bench/drive.py
import os, datetime, re
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/MyDrive/DamageSAR-Bench"
FOLDERS = ["00_admin", "01_data/raw/bright", "01_data/raw/instance_labels", "01_data/external",
           "01_data/derived", "02_models/checkpoints", "02_models/features", "03_notebooks/pipeline",
           "03_notebooks/checks", "03_notebooks/scratch", "04_outputs/runs", "04_outputs/reports"]
for f in FOLDERS:
    os.makedirs(f"{PROJECT}/{f}", exist_ok=True)
def run_dir(name):
    slug = re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")
    p = f"{PROJECT}/04_outputs/runs/{datetime.date.today().isoformat()}_{slug}"
    os.makedirs(p, exist_ok=True)
    return p
RUN = run_dir(NOTEBOOK_NAME)
print("Drive layout ready under", PROJECT)
print("This run saves to:", RUN)

In [ ]:
# Install this repo's package (private repo: needs GH_TOKEN in Colab Secrets, read access to damagesar-bench)
from google.colab import userdata
GH = userdata.get("GH_TOKEN")
!git clone -q https://{GH}@github.com/ThinukaLehan/damagesar-bench.git /content/damagesar-bench 2>/dev/null || (cd /content/damagesar-bench && git pull -q)
!pip install -q -e /content/damagesar-bench
COMMIT = !git -C /content/damagesar-bench rev-parse --short HEAD
COMMIT = COMMIT[0]
# A running Colab kernel does not see an editable install until a restart: put src/ on the path.
import sys, importlib
if "/content/damagesar-bench/src" not in sys.path:
    sys.path.insert(0, "/content/damagesar-bench/src")
importlib.invalidate_caches()
import damagesar_bench
print("damagesar-bench at commit", COMMIT, "->", damagesar_bench.__file__)

In [ ]:
# GPU, Hugging Face token, inputs from p02, p03 and g01
import glob, os, torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as ex:
    raise RuntimeError("HF_TOKEN is missing: add it in Colab -> Secrets (key icon), switch on notebook access, "
                       "then Run all again.") from ex
P02_FEATS = f"{PROJECT}/02_models/features/2026-11-20_dinov3_resolution/features.parquet"
P03 = sorted(glob.glob(f"{PROJECT}/02_models/features/*_controls_imagenet"))
G01 = sorted(glob.glob(f"{PROJECT}/04_outputs/runs/*_g01-gate1-c2-matched-pooling/offsets.csv"))
assert os.path.exists(P02_FEATS), f"missing {P02_FEATS}: run p02 first"
assert P03, "missing the p03 features: run p03 first"
assert G01, "missing the g01 offsets: run g01 first"
P03_RING, P03_FREE = f"{P03[-1]}/dinov3_ring.parquet", f"{P03[-1]}/imagefree.parquet"
assert os.path.exists(P03_RING) and os.path.exists(P03_FREE), "p03 ring / image-free features missing"
print("GPU:", torch.cuda.get_device_name(0), "| p03:", P03[-1], "| g01:", G01[-1])

In [ ]:
# BRIGHT post-event SAR only (~2 GB): labels come from the building table
import zipfile
from huggingface_hub import hf_hub_download
ROOT = "/content/bright"
post = f"{ROOT}/post-event"
if not (os.path.isdir(post) and len(os.listdir(post)) > 4000):
    p = hf_hub_download("Kullervo/BRIGHT", "post-event.zip", repo_type="dataset", local_dir="/content/zips")
    with zipfile.ZipFile(p) as z:
        z.extractall(ROOT)
print(len(os.listdir(post)), "post-event tiles")

In [ ]:
# Building table (development + secondary only), development phase, and the g01 shifts
import datetime, gc
from dataclasses import replace
from datetime import date
import pandas as pd
from damagesar_bench.config import load_events
from damagesar_bench.c2_offsets import matched_shifts
events = load_events("/content/damagesar-bench/configs/events.yaml")
# Exploratory development-phase run: confirmatory labels stay locked here whatever today's date
events = replace(events, freeze_date=date(9999, 12, 31))
B = pd.read_parquet(f"{PROJECT}/01_data/derived/buildings.parquet")
B["damaged"] = B["damaged"].astype("boolean")
use = B[B["role"].isin(["development", "secondary"])].copy()
del B
SHIFTS = matched_shifts(pd.read_csv(G01[-1]), peak_z_min=3.0)
print("tiles with a confident offset (also masked at the shifted position):",
      sum(1 for v in SHIFTS.values() if any(v)))
FEAT_DIR = f"{PROJECT}/02_models/features/{datetime.date.today().isoformat()}_masked_ring"
os.makedirs(FEAT_DIR, exist_ok=True)
print(use[use["excluded_reason"].isna()].groupby("event").size().to_string())

In [ ]:
# Masked tiles -> DINOv3 ViT-B/16 (native) -> 10 m ring, renamed ring10m_masked
from damagesar_bench.controls.masked import masked_name, masked_reader
from damagesar_bench.encoders import get_encoder
from damagesar_bench.encoders.extract import read_bright_tile, write_features
from damagesar_bench.experiments.resolution import extract_at_resolutions, runs_table
enc = get_encoder("dinov3_vitb16")
print("weights:", enc.weights_revision)
reader = masked_reader(lambda t: read_bright_tile(ROOT, t), use, dilate_m=1.0, shifts=SHIFTS)
f, runs = extract_at_resolutions(use, reader, enc, ["ring:10"], [None], "/content/features/dinov3_vitb16_masked",
                                 events, log=print)
MASKED = masked_name(10)
f["pooling"] = MASKED
MASKED_FEATS = str(write_features(f, f"{FEAT_DIR}/dinov3_ring_masked.parquet"))
runs_table(runs).to_csv(f"{RUN}/timing.csv", index=False)
pd.Series({"dinov3_vitb16": enc.weights_revision}, name="weights_revision").to_csv(f"{RUN}/weights_revisions.csv")
del f, enc, use; gc.collect(); torch.cuda.empty_cache()
display(runs_table(runs))

In [ ]:
# Bench run: building vs masked surroundings, unmasked vs masked ring, masked ring vs image-free
from IPython.display import Markdown
from damagesar_bench.run import ComparisonSpec, run_benchmark
D = "dinov3_vitb16@native"
comps = [
    ComparisonSpec((D, "outline"), (D, MASKED), "building vs masked surroundings (DINOv3 ViT-B)"),
    ComparisonSpec((D, "ring10m"), (D, MASKED), "surroundings: unmasked vs masked (DINOv3 ViT-B)"),
    ComparisonSpec((D, MASKED), ("imagefree", "geometry"), "masked surroundings vs image-free"),
]
out = run_benchmark([P02_FEATS, P03_RING, P03_FREE, MASKED_FEATS], comps, events, f"{RUN}/bench_run",
                    n_boot=2000, only_compared=True, event_set="development")
display(Markdown(open(f"{out}/report.md", encoding="utf-8").read()))
m = pd.read_csv(f"{out}/event_metrics.csv")
pv = m.pivot_table(index=["encoder", "pooling"], columns="event", values="auroc"); pv["mean"] = pv.mean(axis=1)
print(pv.round(3).to_string())
c = pd.read_csv(f"{out}/comparisons.csv")
print(c[["label", "estimate", "lower", "upper", "verdict", "sign_flip_p"]].round(4).to_string(index=False))

In [ ]:
# File the executed notebook (with outputs) into this run's folder and 03_notebooks/pipeline/
import json
from google.colab import _message
# Ask the browser for the notebook; it can time out (busy tab, large outputs). Results above are already saved.
nb = None
for attempt in range(3):
    try:
        reply = _message.blocking_request("get_ipynb", timeout_sec=120)
        nb = (reply or {}).get("ipynb")
    except Exception as ex:
        print("attempt", attempt + 1, "failed:", ex)
    if nb:
        break
if not nb:
    print("Could not copy the executed notebook automatically (the results above are already saved).")
    print("Use File -> Save a copy in Drive, then move the copy into", RUN)
else:
    stamp = datetime.date.today().isoformat()
    for target in (f"{RUN}/{NOTEBOOK_NAME}_executed.ipynb",
                   f"{PROJECT}/03_notebooks/pipeline/{stamp}_{NOTEBOOK_NAME}.ipynb"):
        with open(target, "w") as fh:
            json.dump(nb, fh)
        print("saved", target)
    print("run folder:", sorted(os.listdir(RUN)))